[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/04-choropleth-maps.ipynb)

# Choropleth Maps with SocialMapper

A **choropleth map** is a thematic map in which geographic areas are shaded or patterned in proportion to a statistical variable. The word itself comes from the Greek *choros* (area, region) and *plethos* (multitude) -- literally, a "multitude of areas." Choropleth maps are one of the most widely used forms of data visualization in the social sciences, public health, urban planning, and election reporting.

If you have ever seen a map of the United States colored by election results, income levels, or COVID-19 case rates, you have already encountered a choropleth. What makes them powerful is their ability to reveal **spatial patterns** -- clusters of wealth, pockets of aging populations, corridors of density -- that would be invisible in a table of numbers.

In this notebook, you will learn how to create publication-quality choropleth maps using SocialMapper's `create_map` function. We will work with real census data from Portland, Oregon, progressing from basic maps through advanced customization:

1. **Build the data pipeline** -- isochrone, block groups, census data, merge
2. **Create maps for different demographic variables** -- population, income, age
3. **Choose effective colormaps** -- sequential, diverging, and qualitative palettes
4. **Select basemap styles** -- how background tiles provide geographic context
5. **Apply cartographic design principles** -- overlays, statistics, and composition
6. **Export and share** -- GeoJSON, file saving, and interactive HTML maps

## Setup

We begin by importing the four core SocialMapper functions that form our data pipeline, plus IPython display utilities for rendering maps inline.

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper[interactive] @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import create_isochrone, get_census_blocks, get_census_data, create_map
from IPython.display import Image, display, HTML

## What Is a Choropleth Map?

Before we write any code, it is worth understanding what distinguishes a choropleth from other map types.

A choropleth uses **pre-existing geographic boundaries** (census block groups, counties, countries) as the units of analysis. Each unit is filled with a color that represents the value of some variable for that area. This is fundamentally different from:

- **Dot density maps**, which place individual dots to represent counts
- **Heat maps**, which interpolate point data into a continuous surface
- **Proportional symbol maps**, which vary the size of markers

Choropleth maps are the right choice when your data is **aggregated to defined areas** -- exactly what census data provides. The census reports population, income, age, and hundreds of other variables at the block group level, making choropleth mapping a natural fit.

### Strengths and Limitations

**Strengths:**
- Intuitive to read -- humans naturally associate darker colors with "more"
- Effective at revealing regional patterns and spatial clusters
- Work well for normalized data (rates, medians, percentages)

**Limitations to be aware of:**
- Larger areas draw more visual attention regardless of their data values (the **area-size bias**)
- Raw counts (like total population) can be misleading because larger areas naturally have more people
- The choice of color breaks can dramatically change the story the map tells

With these considerations in mind, let us build our first choropleth.

## Step 1 — Build the Data Pipeline

Every choropleth map in SocialMapper follows a four-step pipeline. Understanding each step is essential because the quality of your map depends on the quality of the data feeding into it.

**The pipeline:**

1. `create_isochrone()` -- defines a travel-time boundary around a location
2. `get_census_blocks()` -- finds all census block groups that intersect that boundary
3. `get_census_data()` -- fetches demographic variables for those block groups from the American Community Survey
4. **Merge** -- attaches the census values to each block group's geometry dictionary

Let us walk through each step for Portland, Oregon with a 15-minute driving isochrone.

In [ ]:
# Step 1: Create an isochrone — the area reachable within 15 minutes of driving
# from downtown Portland. This uses the Valhalla routing engine.
iso = create_isochrone("Portland, OR", travel_time=15, travel_mode="drive")

print(f"Isochrone type:     {iso['type']}")
print(f"Location:           {iso['properties']['location']}")
print(f"Travel time:        {iso['properties']['travel_time']} minutes")
print(f"Travel mode:        {iso['properties']['travel_mode']}")
print(f"Area:               {iso['properties']['area_sq_km']:.1f} sq km")

In [ ]:
# Step 2: Get census block groups that fall within the isochrone.
# Block groups are the smallest geographic unit for which the Census Bureau
# publishes detailed demographic data. Each contains 600–3,000 people.
blocks = get_census_blocks(polygon=iso)

print(f"Block groups found: {len(blocks)}")
print(f"First block GEOID:  {blocks[0]['geoid']}")
print(f"Keys per block:     {list(blocks[0].keys())}")

In [ ]:
# Step 3: Fetch census data for our variables of interest.
# SocialMapper translates friendly names like "population" into ACS variable codes
# (e.g., B01003_001E) and queries the Census Bureau API.
variables = ["population", "median_income", "median_age", "housing_units"]
census = get_census_data(iso, variables=variables)

print(f"Location type:      {census.location_type}")
print(f"GEOIDs with data:   {len(census.data)}")
print(f"Variables fetched:  {census.query_info['variables']}")

# Peek at one block group's data
sample_geoid = list(census.data.keys())[0]
print(f"\nSample ({sample_geoid}): {census.data[sample_geoid]}")

In [ ]:
# Step 4: Merge — attach census values to each block group dictionary.
# This creates the flat list-of-dicts structure that create_map expects:
# each dict has a 'geometry' key plus the data columns.
merged_blocks = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        merged_blocks.append({**block, **census.data[geoid]})

print(f"Merged blocks:      {len(merged_blocks)} of {len(blocks)} total")
print(f"Columns available:  {[k for k in merged_blocks[0].keys() if k != 'geometry']}")

We now have a list of dictionaries where each entry contains both a `geometry` (the block group polygon) and numeric columns (`population`, `median_income`, `median_age`, `housing_units`). This is exactly what `create_map` needs.

## Step 2 — Your First Choropleth: Population

The simplest call to `create_map` requires just two arguments: the data and the column to visualize. The function returns a `MapResult` object whose `image_data` field contains PNG bytes that IPython can render inline.

Let us map population -- the most fundamental demographic variable.

In [ ]:
population_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population by Block Group — Portland, OR",
)

print(f"Format:     {population_map.format}")
print(f"Image size: {len(population_map.image_data):,} bytes")
print(f"Metadata:   {population_map.metadata}")

display(Image(data=population_map.image_data))

Notice how the map reveals spatial structure that would be impossible to see in a table. Darker block groups cluster near the urban core where apartment buildings concentrate residents, while lighter areas on the periphery tend to be lower-density single-family neighborhoods.

Also note the legend on the right side -- SocialMapper automatically generates a color bar with the appropriate range and label.

## Step 3 — Mapping Different Variables

The real power of choropleth mapping comes from comparing the spatial patterns of different variables. Does income correlate with density? Where do older residents concentrate? Let us find out.

### Median Household Income

Income maps are among the most commonly produced choropleths. They can reveal the economic geography of a city -- where affluence concentrates, where lower-income neighborhoods cluster, and whether there are sharp boundaries between the two.

In [ ]:
income_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Household Income — Portland, OR",
)
display(Image(data=income_map.image_data))

Compare the income map to the population map above. Areas with high population density do not necessarily have the highest incomes. This kind of multi-variable comparison is one of the most valuable uses of choropleth mapping.

### Median Age

Age distribution maps reveal demographic transitions. College neighborhoods skew young; established suburban areas skew older. Retirement communities stand out clearly.

In [ ]:
age_map = create_map(
    data=merged_blocks,
    column="median_age",
    title="Median Age — Portland, OR",
)
display(Image(data=age_map.image_data))

## Choosing Colormaps

Color is the primary visual channel in a choropleth map, so choosing the right **colormap** (also called a color ramp or palette) is one of the most important design decisions you will make. A poor choice can obscure patterns, mislead readers, or create maps that are inaccessible to colorblind viewers.

There are three families of colormaps, each suited to different types of data:

### Sequential Colormaps

These progress from light to dark (or from one hue to another) and are the default choice for most choropleth data. They communicate a single direction of change: *more* or *less*, *higher* or *lower*.

- **YlGnBu** (yellow-green-blue) -- SocialMapper's default. Works well for population, income, and other count/magnitude variables.
- **viridis** -- Perceptually uniform and colorblind-safe. An excellent general-purpose choice.
- **Blues**, **Greens**, **Oranges** -- Single-hue ramps that are clean and unambiguous.

### Diverging Colormaps

These use two contrasting colors radiating from a neutral midpoint. They are the right choice when your data has a **meaningful center** -- such as zero change, a national average, or a policy threshold.

- **RdYlGn** (red-yellow-green) -- Common for "bad to good" scales, though avoid if colorblind readers are a concern.
- **RdBu** (red-blue) -- Colorblind-friendly diverging palette.
- **coolwarm** -- Subtle diverging palette good for scientific data.

### Qualitative Colormaps

These use distinct, unrelated colors for categorical data. They should **not** be used for numeric data because they imply no ordering.

- **Set2**, **Set3**, **Paired** -- Good for land-use categories or classification labels.

SocialMapper auto-selects a colormap based on your data, but you can override it with the `cmap` parameter. Let us compare three colormaps on the same population data.

In [ ]:
# Compare three colormaps on the same data.
# Each tells a slightly different visual story.

cmap_names = ["YlGnBu", "viridis", "RdYlGn"]

for cmap_name in cmap_names:
    result = create_map(
        data=merged_blocks,
        column="population",
        title=f"Population — {cmap_name} colormap",
        cmap=cmap_name,
    )
    display(Image(data=result.image_data))

**Observations:**

- **YlGnBu** (the default) provides good contrast across the range and reads naturally as "light = low, dark = high."
- **viridis** is perceptually uniform -- equal steps in data produce equal steps in perceived brightness. This makes it the gold standard for scientific visualization and it is fully colorblind-accessible.
- **RdYlGn** is a diverging palette. On population data (which has no natural midpoint), it can be confusing -- green and red imply "good" and "bad," which is not meaningful for population counts. This illustrates why matching your colormap family to your data type matters.

**Rule of thumb:** Use sequential colormaps for counts, rates, and magnitudes. Reserve diverging colormaps for data with a meaningful center value.

## Basemap Context

A **basemap** is the background map layer that provides geographic context -- streets, labels, terrain, water bodies -- beneath your choropleth data. Without a basemap, viewers see colored polygons floating in a void and cannot orient themselves. With one, they can identify neighborhoods, highways, rivers, and other landmarks that help interpret the patterns they see.

SocialMapper supports three CartoDB basemap styles plus the option to disable the basemap entirely:

| Basemap | Description | Best For |
|---|---|---|
| `CartoDB.Voyager` | Warm, labeled streets and features | General-purpose maps; presentations |
| `CartoDB.Positron` | Minimal gray tones, subtle labels | When data should dominate; publications |
| `CartoDB.DarkMatter` | Dark background, bright labels | High contrast; dark-themed dashboards |
| `None` | Plain white background | Clean exports; when basemap distracts |

The default is `CartoDB.Voyager`, which balances readability with visual appeal. Let us compare them.

In [ ]:
# Compare basemap styles on median income data.
basemap_options = [
    ("CartoDB.Voyager", "Voyager (default) — warm, labeled streets"),
    ("CartoDB.Positron", "Positron — minimal gray, subtle labels"),
    ("CartoDB.DarkMatter", "Dark Matter — dark background, bright labels"),
    (None, "No basemap — plain white background"),
]

for basemap_value, description in basemap_options:
    result = create_map(
        data=merged_blocks,
        column="median_income",
        title=f"Median Income — {description}",
        basemap=basemap_value,
    )
    display(Image(data=result.image_data))

Notice how the same data can feel very different depending on the basemap. The Positron style lets the data colors speak most clearly, while the Dark Matter style creates dramatic contrast. For printed reports, Positron or no basemap often work best to conserve ink and maintain clarity.

## Design Principles for Effective Choropleths

Creating a technically correct choropleth is straightforward; creating an *effective* one requires attention to cartographic design principles. Here are the key considerations:

### 1. Use Appropriate Color Scales
Match your colormap family to your data type (sequential for magnitudes, diverging for deviations from a center, qualitative for categories). Prefer perceptually uniform colormaps like viridis for scientific accuracy.

### 2. Include a Legend
SocialMapper always includes one, but when building maps manually, never omit the legend. Without it, your map is just a collection of pretty colors.

### 3. Consider Normalization
Raw counts (total population) can be misleading on choropleths because larger areas naturally contain more people. When possible, use rates or per-capita measures. Median income and median age are already normalized, making them ideal choropleth variables.

### 4. Add Context with Overlays
Boundaries, points of interest, and statistics boxes help viewers interpret what they see. SocialMapper provides all three through the `overlay_boundary`, `overlay_points`, and `show_stats` parameters.

Let us put these principles into practice.

## Overlay: Isochrone Boundary

The block groups returned by `get_census_blocks` often extend beyond the isochrone boundary because the query returns any block group that *intersects* the isochrone, even partially. Overlaying the isochrone boundary makes this clear and helps viewers understand the precise travel-time reach.

Pass the isochrone GeoJSON Feature to the `overlay_boundary` parameter. SocialMapper renders it as a dashed line on top of the choropleth.

In [ ]:
boundary_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population with 15-min Drive Boundary — Portland, OR",
    overlay_boundary=iso,
)
display(Image(data=boundary_map.image_data))

The dashed boundary line shows exactly where the 15-minute driving range ends. Block groups that extend beyond this line are partially reachable -- an important nuance when interpreting the map.

## Overlay: Point Markers

Point overlays let you annotate specific locations on the map -- the center of your analysis, landmarks, facilities, or any other features worth highlighting. Each point is a dictionary with `lat`, `lon`, and an optional `name` label.

This is especially useful when presenting maps to an audience that needs to orient themselves relative to known landmarks.

In [ ]:
portland_landmarks = [
    {"lat": 45.5152, "lon": -122.6784, "name": "Downtown Portland"},
    {"lat": 45.5231, "lon": -122.6765, "name": "Pearl District"},
    {"lat": 45.4895, "lon": -122.6727, "name": "OHSU"},
]

points_map = create_map(
    data=merged_blocks,
    column="population",
    title="Population with Landmarks — Portland, OR",
    overlay_boundary=iso,
    overlay_points=portland_landmarks,
)
display(Image(data=points_map.image_data))

## Statistics Box

A statistics box adds quantitative context directly on the map. This is valuable when the map will be viewed as a standalone image (in a report or presentation) without accompanying text or tables.

SocialMapper offers two modes:

- **Automatic statistics** (`show_stats=True`): Computes and displays summary statistics (count, mean, median, min, max) for the mapped column.
- **Custom statistics** (`stats_dict={...}`): You provide a dictionary of label-value pairs and SocialMapper renders them in a styled box.

### Automatic Statistics

In [ ]:
auto_stats_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Income with Auto Statistics — Portland, OR",
    overlay_boundary=iso,
    show_stats=True,
)
display(Image(data=auto_stats_map.image_data))

### Custom Statistics

For more control, compute your own summary metrics and pass them as a dictionary. This is useful when you want to highlight specific numbers -- total population, number of block groups, or derived metrics like population density.

In [ ]:
import pandas as pd

# Build a DataFrame for easy computation
df = pd.DataFrame(merged_blocks)

custom_stats = {
    "Total Population": f"{df['population'].sum():,.0f}",
    "Block Groups": str(len(df)),
    "Median Income (avg)": f"${df['median_income'].mean():,.0f}",
    "Median Age (avg)": f"{df['median_age'].mean():.1f} years",
}

custom_stats_map = create_map(
    data=merged_blocks,
    column="population",
    title="Portland Demographics — Custom Statistics",
    overlay_boundary=iso,
    show_stats=True,
    stats_dict=custom_stats,
)
display(Image(data=custom_stats_map.image_data))

## Putting It All Together

Let us create a fully composed map that combines all of the design elements we have covered: a meaningful variable, a good colormap, a basemap, boundary overlay, point markers, and custom statistics. This is what a presentation-ready choropleth looks like.

In [ ]:
composed_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Household Income — 15-min Drive from Portland, OR",
    basemap="CartoDB.Positron",
    cmap="viridis",
    overlay_boundary=iso,
    overlay_points=portland_landmarks,
    show_stats=True,
    stats_dict={
        "Block Groups": str(len(merged_blocks)),
        "Income Range": f"${df['median_income'].min():,.0f} – ${df['median_income'].max():,.0f}",
        "Median of Medians": f"${df['median_income'].median():,.0f}",
    },
)
display(Image(data=composed_map.image_data))

## Exporting to GeoJSON

Sometimes you need the underlying geographic data rather than (or in addition to) a rendered image. GeoJSON is a standard format for encoding geographic data structures, widely supported by web mapping libraries (Leaflet, Mapbox GL), GIS software (QGIS, ArcGIS), and data tools (GeoPandas, Turf.js).

Set `export_format="geojson"` and `create_map` returns a `MapResult` whose `geojson_data` field contains a GeoJSON FeatureCollection dictionary. No image is rendered -- this is a pure data export.

In [ ]:
geojson_result = create_map(
    data=merged_blocks,
    column="population",
    export_format="geojson",
)

print(f"Format:           {geojson_result.format}")
print(f"GeoJSON type:     {geojson_result.geojson_data['type']}")
print(f"Feature count:    {len(geojson_result.geojson_data['features'])}")

# Inspect the first feature's properties (excluding the geometry for brevity)
first_feature = geojson_result.geojson_data['features'][0]
print(f"\nFirst feature properties:")
for key, value in first_feature['properties'].items():
    print(f"  {key}: {value}")

## Saving to File

For reports, presentations, or archival, you can save maps directly to disk by passing a `save_path`. SocialMapper writes the file and returns a `MapResult` with the `file_path` field set to the absolute path of the saved file.

Here we use Python's `tempfile` module to create a temporary directory, but in practice you would save to a meaningful location.

In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, "portland_population.png")

    saved_result = create_map(
        data=merged_blocks,
        column="population",
        title="Portland Population — Saved to File",
        save_path=save_path,
        overlay_boundary=iso,
    )

    print(f"Format:     {saved_result.format}")
    print(f"File path:  {saved_result.file_path}")
    print(f"Exists:     {saved_result.file_path.exists()}")
    print(f"File size:  {saved_result.file_path.stat().st_size:,} bytes")

## Interactive HTML Map

Static images are great for reports and publications, but sometimes you want your audience to be able to **pan, zoom, and click** on individual areas to see their values. SocialMapper can generate interactive Leaflet-based HTML maps using the `folium` library.

Set `export_format="html"` and the returned `MapResult` will have an `html_content` field containing a complete HTML document that you can display inline in Jupyter or save as a standalone file.

This requires the optional `folium` dependency. If it is not installed, we catch the `ImportError` gracefully.

In [ ]:
try:
    html_result = create_map(
        data=merged_blocks,
        column="population",
        title="Portland Population (Interactive)",
        export_format="html",
        overlay_boundary=iso,
    )

    print(f"Format:       {html_result.format}")
    print(f"HTML length:  {len(html_result.html_content):,} characters")

    # Render the interactive map inline
    display(HTML(html_result.html_content))

except ImportError:
    print("Interactive maps require folium.")
    print("Install with: pip install 'socialmapper[interactive]'")

## Summary

In this notebook we covered the complete workflow for creating choropleth maps with SocialMapper. Here are the key concepts and their corresponding API calls:

| Concept | How To |
|---|---|
| Create a choropleth map | `create_map(data, column)` |
| Display in Jupyter | `display(Image(data=result.image_data))` |
| Map different variables | Change the `column` parameter |
| Custom colormap | `cmap="viridis"`, `cmap="RdYlGn"`, etc. |
| Change basemap | `basemap="CartoDB.Positron"`, `basemap=None`, etc. |
| Overlay isochrone boundary | `overlay_boundary=iso` |
| Overlay point markers | `overlay_points=[{"lat": ..., "lon": ..., "name": ...}]` |
| Automatic statistics box | `show_stats=True` |
| Custom statistics | `show_stats=True, stats_dict={"Label": "value"}` |
| Export to GeoJSON | `export_format="geojson"` -- access `result.geojson_data` |
| Save to file | `save_path="output.png"` -- access `result.file_path` |
| Interactive HTML map | `export_format="html"` -- access `result.html_content` |

### Design Takeaways

- **Sequential colormaps** (YlGnBu, viridis) work best for most demographic data
- **Diverging colormaps** (RdBu, RdYlGn) should be reserved for data with a meaningful center
- **Basemaps** provide essential geographic context; choose based on your audience and medium
- **Overlays and statistics** transform a simple visualization into a self-contained analytical artifact
- **Normalized variables** (medians, rates, percentages) are more appropriate for choropleths than raw counts

**Next notebook:** [05 — Points of Interest](05-points-of-interest.ipynb)